In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_pagamento")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9dfc58be-1266-4508-9a1d-f20a9bab803b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 171ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [19]:
path = "s3a://bronze/book_pagamento/"
df_book_pagamento = spark.read.parquet(path)
df_book_pagamento.show(5, truncate=False)

+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+--------------------

In [ ]:
+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+-----------------------+---------------------------------------------------+-------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+-------------------+-----------------------+-----------------+---------------------+----------------+---------------------+------------------+--------------------+-----------------------+--------------------+---------------+---------------------+----------------------+
|NUM_CPF    |DAT_STATUS_FATURA |CONTRATO |SEQ_FATURA|NUM_SUB_SEQ_FATURA|NUM_CREDITO_SEQ|DW_TIPO_FATURA|IND_STATUS_FATURA|DW_NUM_CLIENTE|DW_AREA|DW_UN_NEGOCIO|DW_FORMA_PAGAMENTO|VAL_PAGAMENTO_FATURA|DAT_CRIACAO_DW    |DW_BANCO|DW_TIPO_PAGAMENTO|NUM_BANCO_PAGAMENTO|NUM_AGENCIA_PAGAMENTO|NUM_CC_PAGAMENTO|DW_MOTIVO_ESTORNO|VAL_DESCONTO_ITEM|VAL_PAGAMENTO_ITEM|VAL_JUROS_MULTAS_ITEM|VAL_MULTA_EQUIP_ITEM|VAL_MULTA_EQUIP_TOTAL|VAL_MULTA_FID_ITEM|COD_ORIGEM_NETUNO|COD_CONTA_ATIVIDADE|SEQ_ENTIDADE_ATIVIDADE|DAT_CRIACAO_ATIVIDADE|DAT_ATUALIZACAO_ATIVIDADE|COD_LOGIN_OPERADOR_ATIVIDADE|COD_ATIVIDADE|COD_RAZAO_ATIVIDADE|DAT_BAIXA_ATIVIDADE|VAL_BAIXA_ATIVIDADE|DAT_DEPOSITO_ATIVIDADE|COD_FUNDO_ATIVIDADE|COD_BANCO_ATIVIDADE|NUM_CONTA_ATIVIDADE|COD_AGENCIA_ATIVIDADE|SEQ_ENTIDADE_PAGAMENTO|DAT_CRIACAO_PAGAMENTO|DAT_ATUALIZACAO_PAGAMENTO|COD_LOGIN_PAGAMENTO|COD_FORMA_PAGAMENTO|VAL_ORIGINAL_PAGAMENTO|NUM_FATURA_PAGAMENTO|COD_TIPO_PAGAMENTO|DSC_NOME_BANCO_PAGAMENTO|SEQ_ARQUIVO_PAGAMENTO|NUM_PARCELA_PAGAMENTO|NUM_AGRUPADOR_PAGAMENTO|DSC_PAGAMENTO                                      |VAL_ATUAL_PAGAMENTO|COD_METODO_PAGAMENTO|IND_STATUS_PAGAMENTO|DAT_STATUS_PAGAMENTO|COD_ARQUIVO_PAGAMENTO  |COD_NETUNO_PAGAMENTO|DAT_CRIACAO_CREDITO|DAT_ATUALIZACAO_CREDITO|COD_LOGIN_CREDITO|VAL_PAGAMENTO_CREDITO|IND_TIPO_CREDITO|SEQ_PAGAMENTO_CREDITO|SEQ_FATURA_CREDITO|COD_ALOCACAO_CREDITO|COD_DESALOCACAO_CREDITO|SEQ_ENTIDADE_CREDITO|COD_TIPO_FATURA|DAT_ATIVIDADE_CREDITO|DAT_VENCIMENTO_CREDITO|
+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+-----------------------+---------------------------------------------------+-------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+-------------------+-----------------------+-----------------+---------------------+----------------+---------------------+------------------+--------------------+-----------------------+--------------------+---------------+---------------------+----------------------+
|7NYTZUUZX78|14MAR2025:00:00:00|909421358|5         |5                 |5              |-2            |C                |1470063627    |-3     |3            |14                |39.89               |17MAR2025:12:34:41|1694    |30007            |033                |2271                 |-3              |-3               |0                |39.89             |0                    |0                   |0                    |0                 |848400000003989  |177028031          |5                     |14MAR2025:11:11:31   |NULL                     |NULL                        |PYM          |PB                 |14MAR2025:00:00:00 |39.89              |14MAR2025:00:00:00    |NULL               |033                |NULL               |2271                 |5                     |14MAR2025:11:11:31   |15MAR2025:05:26:50       |NULL               |PB                 |39.89                 |177028031005        |P                 |BANESPA                 |5302                 |NULL                 |10930                  |Arquivo Rajada Sequencia: 26155, Registro: 00000114|39.89              |3                   |C                   |15MAR2025:00:00:00  |VXBP20250314B333A69F47B|NULL                |14MAR2025:11:11:31 |NULL                   |NULL             |39.89                |P               |5                    |5                 |PYM                 |NULL                   |5                   |B              |14MAR2025:00:00:00   |20MAR2025:00:00:00    |
|Z98XTT9XUZX|07OCT2023:00:00:00|887727762|15        |24                |17             |-2            |C                |1400318892    |-3     |1            |10                |158.7               |10OCT2023:07:51:24|-3      |30001            |-3                 |-3                   |-3              |-3               |0                |158.7             |4.9                  |0                   |0                    |0                 |NULL             |NULL               |NULL                  |NULL                 |NULL                     |NULL                        |NULL         |NULL               |NULL               |NULL               |NULL                  |NULL               |NULL               |NULL               |NULL                 |NULL                  |NULL                 |NULL                     |NULL               |NULL               |NULL                  |NULL                |NULL              |NULL                    |NULL                 |NULL                 |NULL                   |NULL                                               |NULL               |NULL                |NULL                |NULL                |NULL                   |NULL                |NULL               |NULL                   |NULL             |NULL                 |NULL            |NULL                 |NULL              |NULL                |NULL                   |NULL                |NULL           |NULL                 |NULL                  |
|8W7XU87XN9X|30MAY2024:00:00:00|896411261|13        |13                |13             |-2            |C                |1399351973    |-3     |9            |10                |30.59               |02JUN2024:10:45:11|-3      |30001            |-3                 |-3                   |-3              |-3               |0                |30.59             |0.69                 |0                   |0                    |0                 |NULL             |162750487          |13                    |30MAY2024:11:36:27   |NULL                     |60001                       |PYM          |CA                 |30MAY2024:00:00:00 |30.59              |30MAY2024:00:00:00    |NULL               |NULL               |NULL               |NULL                 |13                    |30MAY2024:11:36:27   |NULL                     |60001              |CA                 |30.59                 |162750487013        |O                 |CPAY-PIX                |NULL                 |NULL                 |NULL                   |NULL                                               |30.59              |NULL                |R                   |30MAY2024:00:00:00  |NULL                   |NULL                |30MAY2024:11:36:27 |NULL                   |60001            |30.59                |P               |13                   |13                |PYM                 |NULL                   |14                  |B              |30MAY2024:00:00:00   |10JUN2024:00:00:00    |
|ZXY9X7ZYXNU|09JUL2024:00:00:00|889406152|23        |23                |25             |-2            |C                |1206089505    |-3     |3            |14                |64.91               |12JUL2024:18:00:39|1368    |30007            |104                |0570                 |-3              |-3               |0                |64.91             |0                    |0                   |0                    |0                 |848200000013027  |154993148          |25                    |09JUL2024:16:18:45   |NULL                     |NULL                        |PYM          |PB                 |09JUL2024:00:00:00 |64.91              |09JUL2024:00:00:00    |NULL               |104                |NULL               |0570                 |25                    |09JUL2024:16:18:45   |10JUL2024:04:06:24       |NULL               |PB                 |64.91                 |154993148023        |P                 |CEF                     |5131                 |NULL                 |63922                  |Arquivo Rajada Sequencia: 27614, Registro: 00000326|64.91              |5                   |C                   |10JUL2024:00:00:00  |331486898              |NULL                |09JUL2024:16:18:45 |NULL                   |NULL             |64.91                |P               |25                   |23                |PYM                 |NULL                   |40                  |B              |09JUL2024:00:00:00   |15JUL2024:00:00:00    |
|NWTWUY9ZXZZ|22FEB2024:00:00:00|888566226|20        |20                |19             |-2            |C                |1201156657    |-3     |3            |14                |68.08               |25FEB2024:10:46:50|1368    |30007            |104                |3444                 |-3              |-3               |0                |68.08             |2.18                 |0                   |0                    |0                 |848100000013168  |NULL               |NULL                  |NULL                 |NULL                     |NULL                        |NULL         |NULL               |NULL               |NULL               |NULL                  |NULL               |NULL               |NULL               |NULL                 |NULL                  |NULL                 |NULL                     |NULL               |NULL               |NULL                  |NULL                |NULL              |NULL                    |NULL                 |NULL                 |NULL                   |NULL                                               |NULL               |NULL                |NULL                |NULL                |NULL                   |NULL                |NULL               |NULL                   |NULL             |NULL                 |NULL            |NULL                 |NULL              |NULL                |NULL                   |NULL                |NULL           |NULL                 |NULL                  |
+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+-----------------------+---------------------------------------------------+-------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+-------------------+-----------------------+-----------------+---------------------+----------------+---------------------+------------------+--------------------+-----------------------+--------------------+---------------

In [20]:
df_book_pagamento.createOrReplaceTempView("raw_00")

In [21]:

raw_00_com_safra = spark.sql("""
    SELECT
        *,
        CAST(
            date_format(
                to_timestamp(DAT_STATUS_FATURA, 'ddMMMyyyy:HH:mm:ss'),
                'yyyyMM'
            ) AS INT
        ) AS SAFRA
    FROM raw_00
""")

raw_00_com_safra.createOrReplaceTempView("raw_00_com_safra")

In [22]:
raw_00_com_safra.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- DAT_STATUS_FATURA: string (nullable = true)
 |-- CONTRATO: string (nullable = true)
 |-- SEQ_FATURA: string (nullable = true)
 |-- NUM_SUB_SEQ_FATURA: string (nullable = true)
 |-- NUM_CREDITO_SEQ: string (nullable = true)
 |-- DW_TIPO_FATURA: string (nullable = true)
 |-- IND_STATUS_FATURA: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: string (nullable = true)
 |-- DW_UN_NEGOCIO: string (nullable = true)
 |-- DW_FORMA_PAGAMENTO: string (nullable = true)
 |-- VAL_PAGAMENTO_FATURA: string (nullable = true)
 |-- DAT_CRIACAO_DW: string (nullable = true)
 |-- DW_BANCO: string (nullable = true)
 |-- DW_TIPO_PAGAMENTO: string (nullable = true)
 |-- NUM_BANCO_PAGAMENTO: string (nullable = true)
 |-- NUM_AGENCIA_PAGAMENTO: string (nullable = true)
 |-- NUM_CC_PAGAMENTO: string (nullable = true)
 |-- DW_MOTIVO_ESTORNO: string (nullable = true)
 |-- VAL_DESCONTO_ITEM: string (nullable = true)
 |-- VAL_PAGAMEN

In [38]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT CONTRATO) as contrato_distintos,
        COUNT(DISTINCT DW_NUM_CLIENTE) as num_tel_distintos
    FROM raw_00_com_safra
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5, truncate=False)

+------+------------+-------------+------------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|contrato_distintos|num_tel_distintos|
+------+------------+-------------+------------------+-----------------+
|202310|869793      |493723       |562289            |562446           |
|202311|869501      |497503       |567229            |567381           |
|202312|929072      |522762       |597176            |597391           |
|202401|923529      |527435       |603103            |603396           |
|202402|943835      |535538       |613121            |613473           |
+------+------------+-------------+------------------+-----------------+
only showing top 5 rows



df_cpf_especifico = spark.sql("""
    SELECT *
    FROM raw_00_com_safra
    WHERE NUM_CPF = 'ZXY9X7ZYXNU'
""")

df_cpf_especifico.show(truncate=False)


In [29]:
print('lista de colunas para tipar')
for col in spark.table("raw_00_com_safra").columns:
    print('cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
cast(NUM_CPF as) as NUM_CPF,
cast(DAT_STATUS_FATURA as) as DAT_STATUS_FATURA,
cast(CONTRATO as) as CONTRATO,
cast(SEQ_FATURA as) as SEQ_FATURA,
cast(NUM_SUB_SEQ_FATURA as) as NUM_SUB_SEQ_FATURA,
cast(NUM_CREDITO_SEQ as) as NUM_CREDITO_SEQ,
cast(DW_TIPO_FATURA as) as DW_TIPO_FATURA,
cast(IND_STATUS_FATURA as) as IND_STATUS_FATURA,
cast(DW_NUM_CLIENTE as) as DW_NUM_CLIENTE,
cast(DW_AREA as) as DW_AREA,
cast(DW_UN_NEGOCIO as) as DW_UN_NEGOCIO,
cast(DW_FORMA_PAGAMENTO as) as DW_FORMA_PAGAMENTO,
cast(VAL_PAGAMENTO_FATURA as) as VAL_PAGAMENTO_FATURA,
cast(DAT_CRIACAO_DW as) as DAT_CRIACAO_DW,
cast(DW_BANCO as) as DW_BANCO,
cast(DW_TIPO_PAGAMENTO as) as DW_TIPO_PAGAMENTO,
cast(NUM_BANCO_PAGAMENTO as) as NUM_BANCO_PAGAMENTO,
cast(NUM_AGENCIA_PAGAMENTO as) as NUM_AGENCIA_PAGAMENTO,
cast(NUM_CC_PAGAMENTO as) as NUM_CC_PAGAMENTO,
cast(DW_MOTIVO_ESTORNO as) as DW_MOTIVO_ESTORNO,
cast(VAL_DESCONTO_ITEM as) as VAL_DESCONTO_ITEM,
cast(VAL_PAGAMENTO_ITEM as) as VAL_PAGAMENT

In [30]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            try_cast(NUM_CPF as STRING) as NUM_CPF,
            try_cast(SAFRA as INT) as SAFRA,
            case 
                when trim(DAT_STATUS_FATURA) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_STATUS_FATURA), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_STATUS_FATURA,
            try_cast(CONTRATO as BIGINT) as CONTRATO,
            try_cast(SEQ_FATURA as INT) as SEQ_FATURA,
            try_cast(NUM_SUB_SEQ_FATURA as INT) as NUM_SUB_SEQ_FATURA,
            try_cast(NUM_CREDITO_SEQ as INT) as NUM_CREDITO_SEQ,
            try_cast(DW_TIPO_FATURA as INT) as DW_TIPO_FATURA,
            try_cast(IND_STATUS_FATURA as STRING) as IND_STATUS_FATURA,
            try_cast(DW_NUM_CLIENTE as BIGINT) as DW_NUM_CLIENTE,
            try_cast(DW_AREA as INT) as DW_AREA,
            try_cast(DW_UN_NEGOCIO as INT) as DW_UN_NEGOCIO,
            try_cast(DW_FORMA_PAGAMENTO as INT) as DW_FORMA_PAGAMENTO,
            try_cast(VAL_PAGAMENTO_FATURA as DECIMAL(10,2)) as VAL_PAGAMENTO_FATURA,
            case 
                when trim(DAT_CRIACAO_DW) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_DW), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_DW,
            try_cast(DW_BANCO as INT) as DW_BANCO,
            try_cast(DW_TIPO_PAGAMENTO as INT) as DW_TIPO_PAGAMENTO,
            try_cast(NUM_BANCO_PAGAMENTO as STRING) as NUM_BANCO_PAGAMENTO,
            try_cast(NUM_AGENCIA_PAGAMENTO as STRING) as NUM_AGENCIA_PAGAMENTO,
            try_cast(NUM_CC_PAGAMENTO as STRING) as NUM_CC_PAGAMENTO,
            try_cast(DW_MOTIVO_ESTORNO as INT) as DW_MOTIVO_ESTORNO,
            try_cast(VAL_DESCONTO_ITEM as DECIMAL(10,2)) as VAL_DESCONTO_ITEM,
            try_cast(VAL_PAGAMENTO_ITEM as DECIMAL(10,2)) as VAL_PAGAMENTO_ITEM,
            try_cast(VAL_JUROS_MULTAS_ITEM as DECIMAL(10,2)) as VAL_JUROS_MULTAS_ITEM,
            try_cast(VAL_MULTA_EQUIP_ITEM as DECIMAL(10,2)) as VAL_MULTA_EQUIP_ITEM,
            try_cast(VAL_MULTA_EQUIP_TOTAL as DECIMAL(10,2)) as VAL_MULTA_EQUIP_TOTAL,
            try_cast(VAL_MULTA_FID_ITEM as DECIMAL(10,2)) as VAL_MULTA_FID_ITEM,
            try_cast(COD_ORIGEM_NETUNO as STRING) as COD_ORIGEM_NETUNO,
            try_cast(COD_CONTA_ATIVIDADE as STRING) as COD_CONTA_ATIVIDADE,
            try_cast(SEQ_ENTIDADE_ATIVIDADE as INT) as SEQ_ENTIDADE_ATIVIDADE,
            case 
                when trim(DAT_CRIACAO_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_ATIVIDADE,
            case 
                when trim(DAT_ATUALIZACAO_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATUALIZACAO_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATUALIZACAO_ATIVIDADE,
            try_cast(COD_LOGIN_OPERADOR_ATIVIDADE as STRING) as COD_LOGIN_OPERADOR_ATIVIDADE,
            try_cast(COD_ATIVIDADE as STRING) as COD_ATIVIDADE,
            try_cast(COD_RAZAO_ATIVIDADE as STRING) as COD_RAZAO_ATIVIDADE,
            case 
                when trim(DAT_BAIXA_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_BAIXA_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_BAIXA_ATIVIDADE,
            try_cast(VAL_BAIXA_ATIVIDADE as DECIMAL(10,2)) as VAL_BAIXA_ATIVIDADE,
            case 
                when trim(DAT_DEPOSITO_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_DEPOSITO_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_DEPOSITO_ATIVIDADE,
            try_cast(COD_FUNDO_ATIVIDADE as STRING) as COD_FUNDO_ATIVIDADE,
            try_cast(COD_BANCO_ATIVIDADE as STRING) as COD_BANCO_ATIVIDADE,
            try_cast(NUM_CONTA_ATIVIDADE as STRING) as NUM_CONTA_ATIVIDADE,
            try_cast(COD_AGENCIA_ATIVIDADE as STRING) as COD_AGENCIA_ATIVIDADE,
            try_cast(SEQ_ENTIDADE_PAGAMENTO as INT) as SEQ_ENTIDADE_PAGAMENTO,
            case 
                when trim(DAT_CRIACAO_PAGAMENTO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_PAGAMENTO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_PAGAMENTO,
            case 
                when trim(DAT_ATUALIZACAO_PAGAMENTO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATUALIZACAO_PAGAMENTO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATUALIZACAO_PAGAMENTO,
            try_cast(COD_LOGIN_PAGAMENTO as STRING) as COD_LOGIN_PAGAMENTO,
            try_cast(COD_FORMA_PAGAMENTO as STRING) as COD_FORMA_PAGAMENTO,
            try_cast(VAL_ORIGINAL_PAGAMENTO as DECIMAL(10,2)) as VAL_ORIGINAL_PAGAMENTO,
            try_cast(NUM_FATURA_PAGAMENTO as STRING) as NUM_FATURA_PAGAMENTO,
            try_cast(COD_TIPO_PAGAMENTO as STRING) as COD_TIPO_PAGAMENTO,
            try_cast(DSC_NOME_BANCO_PAGAMENTO as STRING) as DSC_NOME_BANCO_PAGAMENTO,
            try_cast(SEQ_ARQUIVO_PAGAMENTO as INT) as SEQ_ARQUIVO_PAGAMENTO,
            try_cast(NUM_PARCELA_PAGAMENTO as STRING) as NUM_PARCELA_PAGAMENTO,
            try_cast(NUM_AGRUPADOR_PAGAMENTO as STRING) as NUM_AGRUPADOR_PAGAMENTO,
            try_cast(DSC_PAGAMENTO as STRING) as DSC_PAGAMENTO,
            try_cast(VAL_ATUAL_PAGAMENTO as DECIMAL(10,2)) as VAL_ATUAL_PAGAMENTO,
            try_cast(COD_METODO_PAGAMENTO as INT) as COD_METODO_PAGAMENTO,
            try_cast(IND_STATUS_PAGAMENTO as STRING) as IND_STATUS_PAGAMENTO,
            case 
                when trim(DAT_STATUS_PAGAMENTO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_STATUS_PAGAMENTO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_STATUS_PAGAMENTO,
            try_cast(COD_ARQUIVO_PAGAMENTO as STRING) as COD_ARQUIVO_PAGAMENTO,
            try_cast(COD_NETUNO_PAGAMENTO as STRING) as COD_NETUNO_PAGAMENTO,
            case 
                when trim(DAT_CRIACAO_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_CREDITO,
            case 
                when trim(DAT_ATUALIZACAO_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATUALIZACAO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATUALIZACAO_CREDITO,
            try_cast(COD_LOGIN_CREDITO as STRING) as COD_LOGIN_CREDITO,
            try_cast(VAL_PAGAMENTO_CREDITO as DECIMAL(10,2)) as VAL_PAGAMENTO_CREDITO,
            try_cast(IND_TIPO_CREDITO as STRING) as IND_TIPO_CREDITO,
            try_cast(SEQ_PAGAMENTO_CREDITO as INT) as SEQ_PAGAMENTO_CREDITO,
            try_cast(SEQ_FATURA_CREDITO as INT) as SEQ_FATURA_CREDITO,
            try_cast(COD_ALOCACAO_CREDITO as STRING) as COD_ALOCACAO_CREDITO,
            try_cast(COD_DESALOCACAO_CREDITO as STRING) as COD_DESALOCACAO_CREDITO,
            try_cast(SEQ_ENTIDADE_CREDITO as INT) as SEQ_ENTIDADE_CREDITO,
            try_cast(COD_TIPO_FATURA as STRING) as COD_TIPO_FATURA,
            case 
                when trim(DAT_ATIVIDADE_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATIVIDADE_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATIVIDADE_CREDITO,
            case 
                when trim(DAT_VENCIMENTO_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_VENCIMENTO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_VENCIMENTO_CREDITO,
            {pdthproc} as DATPROC

        from
            raw_00_com_safra
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

21829628

In [31]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- DAT_STATUS_FATURA: timestamp (nullable = true)
 |-- CONTRATO: long (nullable = true)
 |-- SEQ_FATURA: integer (nullable = true)
 |-- NUM_SUB_SEQ_FATURA: integer (nullable = true)
 |-- NUM_CREDITO_SEQ: integer (nullable = true)
 |-- DW_TIPO_FATURA: integer (nullable = true)
 |-- IND_STATUS_FATURA: string (nullable = true)
 |-- DW_NUM_CLIENTE: long (nullable = true)
 |-- DW_AREA: integer (nullable = true)
 |-- DW_UN_NEGOCIO: integer (nullable = true)
 |-- DW_FORMA_PAGAMENTO: integer (nullable = true)
 |-- VAL_PAGAMENTO_FATURA: decimal(10,2) (nullable = true)
 |-- DAT_CRIACAO_DW: timestamp (nullable = true)
 |-- DW_BANCO: integer (nullable = true)
 |-- DW_TIPO_PAGAMENTO: integer (nullable = true)
 |-- NUM_BANCO_PAGAMENTO: string (nullable = true)
 |-- NUM_AGENCIA_PAGAMENTO: string (nullable = true)
 |-- NUM_CC_PAGAMENTO: string (nullable = true)
 |-- DW_MOTIVO_ESTORNO: integer (nullable = true)
 |-- VAL

In [33]:
lake.show(5)

+-----------+------+-------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+-------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+-----------

In [ ]:
+-----------+------+-------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+-------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+-----------------------+--------------------+-------------------+--------------------+--------------------+--------------------+---------------------+--------------------+-------------------+-----------------------+-----------------+---------------------+----------------+---------------------+------------------+--------------------+-----------------------+--------------------+---------------+---------------------+----------------------+--------------+
|    NUM_CPF| SAFRA|  DAT_STATUS_FATURA| CONTRATO|SEQ_FATURA|NUM_SUB_SEQ_FATURA|NUM_CREDITO_SEQ|DW_TIPO_FATURA|IND_STATUS_FATURA|DW_NUM_CLIENTE|DW_AREA|DW_UN_NEGOCIO|DW_FORMA_PAGAMENTO|VAL_PAGAMENTO_FATURA|     DAT_CRIACAO_DW|DW_BANCO|DW_TIPO_PAGAMENTO|NUM_BANCO_PAGAMENTO|NUM_AGENCIA_PAGAMENTO|NUM_CC_PAGAMENTO|DW_MOTIVO_ESTORNO|VAL_DESCONTO_ITEM|VAL_PAGAMENTO_ITEM|VAL_JUROS_MULTAS_ITEM|VAL_MULTA_EQUIP_ITEM|VAL_MULTA_EQUIP_TOTAL|VAL_MULTA_FID_ITEM|COD_ORIGEM_NETUNO|COD_CONTA_ATIVIDADE|SEQ_ENTIDADE_ATIVIDADE|DAT_CRIACAO_ATIVIDADE|DAT_ATUALIZACAO_ATIVIDADE|COD_LOGIN_OPERADOR_ATIVIDADE|COD_ATIVIDADE|COD_RAZAO_ATIVIDADE|DAT_BAIXA_ATIVIDADE|VAL_BAIXA_ATIVIDADE|DAT_DEPOSITO_ATIVIDADE|COD_FUNDO_ATIVIDADE|COD_BANCO_ATIVIDADE|NUM_CONTA_ATIVIDADE|COD_AGENCIA_ATIVIDADE|SEQ_ENTIDADE_PAGAMENTO|DAT_CRIACAO_PAGAMENTO|DAT_ATUALIZACAO_PAGAMENTO|COD_LOGIN_PAGAMENTO|COD_FORMA_PAGAMENTO|VAL_ORIGINAL_PAGAMENTO|NUM_FATURA_PAGAMENTO|COD_TIPO_PAGAMENTO|DSC_NOME_BANCO_PAGAMENTO|SEQ_ARQUIVO_PAGAMENTO|NUM_PARCELA_PAGAMENTO|NUM_AGRUPADOR_PAGAMENTO|       DSC_PAGAMENTO|VAL_ATUAL_PAGAMENTO|COD_METODO_PAGAMENTO|IND_STATUS_PAGAMENTO|DAT_STATUS_PAGAMENTO|COD_ARQUIVO_PAGAMENTO|COD_NETUNO_PAGAMENTO|DAT_CRIACAO_CREDITO|DAT_ATUALIZACAO_CREDITO|COD_LOGIN_CREDITO|VAL_PAGAMENTO_CREDITO|IND_TIPO_CREDITO|SEQ_PAGAMENTO_CREDITO|SEQ_FATURA_CREDITO|COD_ALOCACAO_CREDITO|COD_DESALOCACAO_CREDITO|SEQ_ENTIDADE_CREDITO|COD_TIPO_FATURA|DAT_ATIVIDADE_CREDITO|DAT_VENCIMENTO_CREDITO|       DATPROC|
+-----------+------+-------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+-------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+-----------------------+--------------------+-------------------+--------------------+--------------------+--------------------+---------------------+--------------------+-------------------+-----------------------+-----------------+---------------------+----------------+---------------------+------------------+--------------------+-----------------------+--------------------+---------------+---------------------+----------------------+--------------+
|7NYTZUUZX78|202503|2025-03-14 00:00:00|909421358|         5|                 5|              5|            -2|                C|    1470063627|     -3|            3|                14|               39.89|2025-03-17 12:34:41|    1694|            30007|                033|                 2271|              -3|               -3|             0.00|             39.89|                 0.00|                0.00|                 0.00|              0.00|  848400000003989|          177028031|                     5|  2025-03-14 11:11:31|                     NULL|                        NULL|          PYM|                 PB|2025-03-14 00:00:00|              39.89|   2025-03-14 00:00:00|               NULL|                033|               NULL|                 2271|                     5|  2025-03-14 11:11:31|      2025-03-15 05:26:50|               NULL|                 PB|                 39.89|        177028031005|                 P|                 BANESPA|                 5302|                 NULL|                  10930|Arquivo Rajada Se...|              39.89|                   3|                   C| 2025-03-15 00:00:00| VXBP20250314B333A...|                NULL|2025-03-14 11:11:31|                   NULL|             NULL|                39.89|               P|                    5|                 5|                 PYM|                   NULL|                   5|              B|  2025-03-14 00:00:00|   2025-03-20 00:00:00|20251230074024|
|Z98XTT9XUZX|202310|2023-10-07 00:00:00|887727762|        15|                24|             17|            -2|                C|    1400318892|     -3|            1|                10|              158.70|2023-10-10 07:51:24|      -3|            30001|                 -3|                   -3|              -3|               -3|             0.00|            158.70|                 4.90|                0.00|                 0.00|              0.00|             NULL|               NULL|                  NULL|                 NULL|                     NULL|                        NULL|         NULL|               NULL|               NULL|               NULL|                  NULL|               NULL|               NULL|               NULL|                 NULL|                  NULL|                 NULL|                     NULL|               NULL|               NULL|                  NULL|                NULL|              NULL|                    NULL|                 NULL|                 NULL|                   NULL|                NULL|               NULL|                NULL|                NULL|                NULL|                 NULL|                NULL|               NULL|                   NULL|             NULL|                 NULL|            NULL|                 NULL|              NULL|                NULL|                   NULL|                NULL|           NULL|                 NULL|                  NULL|20251230074024|
|8W7XU87XN9X|202405|2024-05-30 00:00:00|896411261|        13|                13|             13|            -2|                C|    1399351973|     -3|            9|                10|               30.59|2024-06-02 10:45:11|      -3|            30001|                 -3|                   -3|              -3|               -3|             0.00|             30.59|                 0.69|                0.00|                 0.00|              0.00|             NULL|          162750487|                    13|  2024-05-30 11:36:27|                     NULL|                       60001|          PYM|                 CA|2024-05-30 00:00:00|              30.59|   2024-05-30 00:00:00|               NULL|               NULL|               NULL|                 NULL|                    13|  2024-05-30 11:36:27|                     NULL|              60001|                 CA|                 30.59|        162750487013|                 O|                CPAY-PIX|                 NULL|                 NULL|                   NULL|                NULL|              30.59|                NULL|                   R| 2024-05-30 00:00:00|                 NULL|                NULL|2024-05-30 11:36:27|                   NULL|            60001|                30.59|               P|                   13|                13|                 PYM|                   NULL|                  14|              B|  2024-05-30 00:00:00|   2024-06-10 00:00:00|20251230074024|
|ZXY9X7ZYXNU|202407|2024-07-09 00:00:00|889406152|        23|                23|             25|            -2|                C|    1206089505|     -3|            3|                14|               64.91|2024-07-12 18:00:39|    1368|            30007|                104|                 0570|              -3|               -3|             0.00|             64.91|                 0.00|                0.00|                 0.00|              0.00|  848200000013027|          154993148|                    25|  2024-07-09 16:18:45|                     NULL|                        NULL|          PYM|                 PB|2024-07-09 00:00:00|              64.91|   2024-07-09 00:00:00|               NULL|                104|               NULL|                 0570|                    25|  2024-07-09 16:18:45|      2024-07-10 04:06:24|               NULL|                 PB|                 64.91|        154993148023|                 P|                     CEF|                 5131|                 NULL|                  63922|Arquivo Rajada Se...|              64.91|                   5|                   C| 2024-07-10 00:00:00|            331486898|                NULL|2024-07-09 16:18:45|                   NULL|             NULL|                64.91|               P|                   25|                23|                 PYM|                   NULL|                  40|              B|  2024-07-09 00:00:00|   2024-07-15 00:00:00|20251230074024|
|NWTWUY9ZXZZ|202402|2024-02-22 00:00:00|888566226|        20|                20|             19|            -2|                C|    1201156657|     -3|            3|                14|               68.08|2024-02-25 10:46:50|    1368|            30007|                104|                 3444|              -3|               -3|             0.00|             68.08|                 2.18|                0.00|                 0.00|              0.00|  848100000013168|               NULL|                  NULL|                 NULL|                     NULL|                        NULL|         NULL|               NULL|               NULL|               NULL|                  NULL|               NULL|               NULL|               NULL|                 NULL|                  NULL|                 NULL|                     NULL|               NULL|               NULL|                  NULL|                NULL|              NULL|                    NULL|                 NULL|                 NULL|                   NULL|                NULL|               NULL|                NULL|                NULL|                NULL|                 NULL|                NULL|               NULL|                   NULL|             NULL|                 NULL|            NULL|                 NULL|              NULL|                NULL|                   NULL|                NULL|           NULL|                 NULL|                  NULL|20251230074024|
+-----------+------+-------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+-------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+-----------------------+--------------------+-------------------+--------------------+--------------------+--------------------+---------------------+--------------------+-------------------+-----------------------+-----------------+---------------------+----------------+---------------------+------------------+--------------------+-----------------------+--------------------+---------------+---------------------+----------------------

In [39]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, DAT_STATUS_FATURA, CONTRATO, NUM_SUB_SEQ_FATURA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

21829628

In [40]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_pagamento/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.DAT_STATUS_FATURA = s.DAT_STATUS_FATURA
            AND t.CONTRATO = s.CONTRATO
            AND t.NUM_SUB_SEQ_FATURA = s.NUM_SUB_SEQ_FATURA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [41]:
spark.stop()